In [3]:
import os
import glob
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


# ============================================================
# 1) LLM + EMBEDDINGS
# ============================================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# ============================================================
# 2) MESSAGE HISTORY STORE
# ============================================================
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

SESSION_ID = "pdf_faiss_session"


# ============================================================
# 3) LOAD CSV INTO CONVERSATION HISTORY (PRELOAD) ✅ FIXED
# ============================================================
def preload_history_from_csv(
    session_id: str,
    csv_path: str,
    max_rows: int | None = None,
    clear_existing: bool = False,
    debug: bool = True,
) -> ChatMessageHistory:
    """
    Loads Question/Answer pairs into the ChatMessageHistory for the given session_id.

    Fixes:
    - Returns the history so caller can verify/use it.
    - Optional clear_existing to avoid silent duplicates or stale state.
    - Debug shows count + sample messages proving it loaded.
    """
    df = pd.read_csv(csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"CSV must contain 'Question' and 'Answer'. Found: {list(df.columns)}")

    history = get_history(session_id)

    if clear_existing:
        history.clear()

    # Load rows
    loaded_pairs = 0
    for i, row in df.iterrows():
        if max_rows is not None and loaded_pairs >= max_rows:
            break
        history.add_user_message(str(row["Question"]))
        history.add_ai_message(str(row["Answer"]))
        loaded_pairs += 1

    # ✅ PROOF it loaded into the SAME session history used by chains
    if debug:
        msg_count = len(history.messages)
        print(f"[preload_history_from_csv] Loaded {loaded_pairs} Q/A pairs -> {msg_count} messages in session '{session_id}'")
        if msg_count >= 2:
            print("[preload_history_from_csv] Sample:")
            print("  USER:", history.messages[0].content[:120])
            print("  AI  :", history.messages[1].content[:120])

    return history


CSV_PATH = r"C:\Users\surya.adatravu\Documents\CONV_WINDOW_LLM_ANALYSIS\RA_FSM_QA.csv"
preload_history_from_csv(SESSION_ID, CSV_PATH, max_rows=None, clear_existing=True, debug=True)


# ============================================================
# 4) LOAD PDF DOCUMENTS
# ============================================================
def load_pdfs_from_folder(folder_path: str):
    pdf_paths = sorted(glob.glob(os.path.join(folder_path, "*.pdf")))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {folder_path}")

    all_docs = []
    for p in pdf_paths:
        loader = PyPDFLoader(p)
        docs = loader.load()
        for d in docs:
            d.metadata["source_file"] = os.path.basename(p)
        all_docs.extend(docs)

    return all_docs


# ============================================================
# 5) CHUNKING
# ============================================================
def chunk_documents(docs, chunk_size=900, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
    return chunks


# ============================================================
# 6) BUILD / LOAD FAISS INDEX
# ============================================================
def build_or_load_faiss(chunks, index_dir):
    os.makedirs(index_dir, exist_ok=True)
    faiss_path = os.path.join(index_dir, "index.faiss")
    pkl_path = os.path.join(index_dir, "index.pkl")

    if os.path.exists(faiss_path) and os.path.exists(pkl_path):
        return FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)

    vs = FAISS.from_documents(chunks, embeddings)
    vs.save_local(index_dir)
    return vs


# ============================================================
# 7) CONTEXT-AWARE QUERY REWRITE (USES HISTORY)
# ============================================================
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone search query.\n"
     "Use chat history only to resolve references/pronouns.\n"
     "Return ONLY the rewritten query."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

rewrite_chain = rewrite_prompt | llm

rewrite_with_history = RunnableWithMessageHistory(
    rewrite_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ============================================================
# 8) RAG ANSWER PROMPT (INCLUDES HISTORY)
# ============================================================
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "You must ONLY answer using the provided CONTEXT.\n"
     "Chat history may help interpret the user's question, but it is NOT a knowledge source.\n"
     "If the answer is not in the context, say exactly:\n"
     "\"I don't know from provided knowledge.\""),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "CONTEXT:\n{context}\n\n"
     "QUESTION:\n{question}\n\n"
     "Answer using ONLY the context.")
])

rag_chain = rag_prompt | llm

rag_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)


# ============================================================
# 9) VALIDATION
# ============================================================
def similarity_score(text1, text2):
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])

def validate_answer(answer, retrieved_docs, threshold=0.80):
    best = -1.0
    for d in retrieved_docs:
        best = max(best, similarity_score(answer, d.page_content))
    return best >= threshold, best


# ============================================================
# 10) ASK FUNCTION
# ============================================================
def ask(question, vectorstore, top_k=5, distance_threshold=0.75, validation_threshold=0.80, debug=False):
    # Prove history exists at time of asking
    if debug:
        h = get_history(SESSION_ID)
        print(f"[ask] session '{SESSION_ID}' history messages: {len(h.messages)}")

    rewritten = rewrite_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    ).content.strip()

    docs_with_scores = vectorstore.similarity_search_with_score(rewritten, k=top_k)
    if not docs_with_scores:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge."}

    top_distance = float(docs_with_scores[0][1])
    if top_distance > distance_threshold:
        return {"question": question, "rewritten_query": rewritten, "answer": "I don't know from provided knowledge."}

    retrieved_docs = [d for d, _ in docs_with_scores]
    context = "\n\n---\n\n".join(
        [f"[file={d.metadata.get('source_file','?')} page={d.metadata.get('page','?')}]\n{d.page_content}"
         for d in retrieved_docs]
    )

    resp = rag_with_history.invoke(
        {"question": question, "context": context},
        config={"configurable": {"session_id": SESSION_ID}}
    )
    answer = resp.content.strip()

    if answer != "I don't know from provided knowledge.":
        ok, score = validate_answer(answer, retrieved_docs, threshold=validation_threshold)
        if not ok:
            return {
                "question": question,
                "rewritten_query": rewritten,
                "answer": "Rejected: hallucination detected",
                "retrieval_distance": top_distance,
                "validation_score": score,
            }

    return {
        "question": question,
        "rewritten_query": rewritten,
        "answer": answer,
        "retrieval_distance": top_distance,
    }


# ============================================================
# 11) MAIN
# ============================================================
if __name__ == "__main__":
    PDF_FOLDER = r"C:\Users\surya.adatravu\Documents\ContextRAG\pdfs_folder"
    INDEX_DIR = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"

    docs = load_pdfs_from_folder(PDF_FOLDER)
    chunks = chunk_documents(docs)
    vs = build_or_load_faiss(chunks, INDEX_DIR)

    #print(ask("provide details of all states in finite state machine", vs, debug=True))
    print(ask("provide details of all states in finite state machine", vs))
    print(ask("provide details of confidence threshold levels", vs))
    print(ask("and what about relevance threshold?", vs))


[preload_history_from_csv] Loaded 50 Q/A pairs -> 100 messages in session 'pdf_faiss_session'
[preload_history_from_csv] Sample:
  USER: What is RA–FSM?
  AI  : RA–FSM is a modular, GPT-based research assistant that uses a finite-state control loop (Relevance → Confidence → Knowle
{'question': 'provide details of all states in finite state machine', 'rewritten_query': "The finite state machine (FSM) in RA–FSM consists of three core states: \n\n1. **Relevance State**: This state checks if the user's question falls within the agent's domain of expertise. It ensures that the system only attempts to answer questions it is equipped to handle.\n\n2. **Confidence State**: In this state, the system evaluates whether it can confidently answer the question based on its internal context. It assesses the confidence score to determine if the answer can be provided reliably.\n\n3. **Knowledge State**: This state is responsible for retrieving external information, synthesizing answers, and appending 